# ChangeFormer Architecture

This notebook introduces the original ChangeFormer architecture
for bi-temporal remote-sensing image change detection.

ChangeFormer is a Transformer-based Siamese network proposed by
Wele Gedara Chaminda Bandara and Vishal M. Patel.

The architecture uses a hierarchically structured Transformer
encoder and an MLP-based decoder to capture multi-scale and
long-range information from bi-temporal satellite images.

The finalized architecture used in the official implementation
is ChangeFormerV6.

## Objective

The objective of this notebook is to:

1. Load the original ChangeFormer implementation.
2. Instantiate ChangeFormerV6.
3. Verify the model architecture.
4. Check the number of trainable parameters.
5. Perform a GPU forward-pass sanity test using 256×256 images.

Training will be performed in the following notebook.

In [18]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", DEVICE)

Using: cuda


In [19]:
import sys
from pathlib import Path

CHANGEFORMER_ROOT = Path("../ChangeFormer").resolve()

sys.path.insert(0, str(CHANGEFORMER_ROOT))

print(CHANGEFORMER_ROOT)

C:\Users\balas\capstone\ChangeFormer


In [20]:
from models.networks import define_G

print("ChangeFormer repository loaded")

ChangeFormer repository loaded


In [21]:
import inspect
from models.networks import define_G

print(inspect.signature(define_G))

(args, init_type='normal', init_gain=0.02, gpu_ids=[])


In [22]:
from types import SimpleNamespace

args = SimpleNamespace(
    net_G="ChangeFormerV6",
    embed_dim=256,
    output_nc=2
)

print(args)

namespace(net_G='ChangeFormerV6', embed_dim=256, output_nc=2)


In [23]:
model = define_G(
    args,
    gpu_ids=[]
)

model = model.to(DEVICE)

print(model.__class__.__name__)

initialize network with normal
ChangeFormerV6


In [24]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

Total parameters     : 41,026,674
Trainable parameters : 41,026,674


In [25]:
import torch

torch.cuda.empty_cache()

model.eval()

before = torch.randn(
    1, 3, 256, 256,
    device=DEVICE
)

after = torch.randn(
    1, 3, 256, 256,
    device=DEVICE
)

with torch.no_grad():

    output = model(
        before,
        after
    )

print("Before :", before.shape)
print("After  :", after.shape)

Before : torch.Size([1, 3, 256, 256])
After  : torch.Size([1, 3, 256, 256])


In [26]:
print(type(output))
print("Number of outputs:", len(output))

for i, out in enumerate(output):
    print(f"Output {i}:", out.shape)

<class 'list'>
Number of outputs: 5
Output 0: torch.Size([1, 2, 8, 8])
Output 1: torch.Size([1, 2, 16, 16])
Output 2: torch.Size([1, 2, 32, 32])
Output 3: torch.Size([1, 2, 64, 64])
Output 4: torch.Size([1, 2, 256, 256])


In [27]:
final_output = output[-1]

print("Final output shape:", final_output.shape)
print("Expected shape    :", (1, 2, 256, 256))

Final output shape: torch.Size([1, 2, 256, 256])
Expected shape    : (1, 2, 256, 256)


# Conclusion

The original ChangeFormerV6 architecture was successfully integrated into the
research environment without modifying its core architecture.

The model contains 41,026,674 trainable parameters and successfully performs
a forward pass on the GPU using 256 × 256 bi-temporal satellite images.

The model produces five multi-scale predictions:

- 8 × 8
- 16 × 16
- 32 × 32
- 64 × 64
- 256 × 256

The final prediction has the shape:

    [1, 2, 256, 256]

where the two output channels represent the two change-detection classes:
unchanged and changed.

This confirms that the ChangeFormerV6 implementation is compatible with the
current experimental environment and can be used for the next stage of the
project.

The next notebook will focus on adapting the LEVIR-CD dataset targets and
training the original ChangeFormerV6 model.